* After Chapter 7.1 motivates locality and weight sharing, Chapter 7.2 opens the actual operation.

* A convolutional layer **turns a small local pattern detector into a feature map by applying the same learned weights across many local windows**.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain a kernel as a learned local pattern detector
- implement two-dimensional cross-correlation with tiny tensors
- verify the manual operation against `nn.Conv2d`
- learn a tiny edge detector through gradient descent
- debug channel-shape errors in `Conv2d`

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def corr2d(X, K): # Hand-writen version of nn.functional.conv2d()
    h, w = K.shape
    out_h = X.shape[0] - h + 1 # Output height = input height − kernel height + 1
    out_w = X.shape[1] - w + 1 # Output width = input width − kernel width + 1
    Y = torch.zeros(out_h, out_w, dtype=X.dtype) # Y has shape (out_h, out_w)
    for i in range(out_h):
        for j in range(out_w):
            window = X[i:i+h, j:j+w] # Extract an h × w window (shapes of K) from X
            Y[i, j] = (window * K).sum() # Calculate the value that belongs at position (i, j) in Y with the sum of element-wise products of window and K
    return Y

# 7.2.0 The Problem This Notebook Solves

Chapter 7.1 said a CNN reuses local detectors. This notebook answers:

```text
what does one local detector actually compute?
```

* A kernel is a small grid of weights.
* At each valid location, it aligns with a window of the input.
* Matching entries are multiplied, and all products are added into one output value.
* Sliding the kernel over all valid locations creates a feature map.

Plain-English meaning:

- The kernel asks a local question.
- The feature map records where the answer is strong.
- The same question is asked at many positions.

In deep learning libraries, this operation is usually cross-correlation, even though the layer is called convolution.

The difference is whether the kernel is flipped before sliding. For learned kernels, this naming mismatch usually does not matter because training can learn whichever orientation is useful.

For hand-written kernels, it matters enough to be precise.

The handoff from this notebook to later CNN sections is direct:

- padding changes which windows are valid near the border
- stride changes which window positions are visited
- channels let a kernel combine several measurements at each location
- pooling summarizes feature maps after convolution

# 7.2.1 Manual Cross-Correlation

The manual implementation is intentionally tiny. It is not for speed. It is for ownership of the idea.

For each output position:

```text
choose a window from X
multiply the window by K entry by entry
sum the products
write that scalar into Y
```

The output is smaller than the input because the kernel only visits positions where it fully fits inside the input. No padding is used yet.

Before running the cell, inspect the top-left window:

```text
[[0, 1],
 [3, 4]]
```

With kernel:

```text
[[1, 0],
 [0, -1]]
```

The response is `0 * 1 + 1 * 0 + 3 * 0 + 4 * -1 = -4`.

In [ ]:
X = torch.arange(9).reshape(3, 3)
K = torch.tensor([
    [1.0, 0.0],
    [0.0, -1.0],
]) # shape (2, 2)

Y = corr2d(X, K) # out_h = 3 - 2 + 1 = 2; out_w = 3 - 2 + 1 = 2

print(Y)

expected = torch.tensor([[-4.0, -4.0], [-4.0, -4.0]])
assert torch.equal(Y, expected)
assert shape(Y) == (2, 2)

tensor([[-4, -4],
        [-4, -4]])


# 7.2.2 Verify Against `nn.Conv2d`

Manual code proves the operation. `nn.Conv2d` is the production-shaped PyTorch module.

PyTorch image batches are shaped:

```text
batch, channels, height, width
```

That extra batch dimension matters because models usually process many examples at once.

The channel dimension matters because real images and hidden feature maps carry multiple measurements per location.

In this cell, there is one example, one input channel, one output channel, and a 2 by 2 kernel.

We manually copy the kernel values into the layer's weight so the PyTorch result should match `corr2d` exactly.

In [ ]:
conv = nn.Conv2d(1, 1, kernel_size=(2, 2), bias=False)
with torch.no_grad():
    conv.weight[:] = K.reshape(1, 1, 2, 2)

X4 = X.reshape(1, 1, 3, 3).float()
Y4 = conv(X4)

print("manual:")
print(Y)
print("conv2d:")
print(Y4[0, 0])

assert torch.allclose(Y4[0, 0], Y.float())
assert shape(Y4) == (1, 1, 2, 2)

manual:
tensor([[-4, -4],
        [-4, -4]])
conv2d:
tensor([[-4., -4.],
        [-4., -4.]], grad_fn=<SelectBackward0>)


# 7.2.3 A Hand-Written Edge Detector

An edge is a sharp local change. The kernel `[1, -1]` asks a simple local question:

```text
is the left value larger than the right value?
```

* If the left pixel is bright and the right pixel is dark, the response is positive.
* If both are similar, the response is near zero.
* If the direction is reversed, the response is negative.

This is a handcrafted detector. It is useful pedagogically because the kernel has an interpretable meaning.

Real CNNs usually learn kernels from data, and early learned kernels often become edge-like as useful low-level image features.

In [ ]:
X = torch.tensor([
    [1.0, 1.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
])
K = torch.tensor([[1.0, -1.0]]) # Shape (1, 2)

Y = corr2d(X, K) # out_h = 3 - 1 + 1 = 3; out_w = 4 - 2 + 1 = 3
print(Y)

assert shape(Y) == (3, 3)
assert torch.equal(Y[:, 1], torch.ones(3))

tensor([[0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.]])


# 7.2.4 Learn the Kernel Instead of Hand-Picking It

The hand-written edge detector shows what a kernel can mean. But manually designing every useful visual feature does not scale.

The deep learning move is:

```text
make the kernel a parameter
define a loss
use gradients to change the kernel
```

This cell gives the model an artificial target feature map: respond strongly at the vertical edge and weakly elsewhere.

The target is not a real classification label; it is a **tiny supervised signal designed to expose the learning mechanism**.

The important theoretical point is that **convolutional layers do not need humans to write the filters**.

The architecture constrains the form of the computation - local and shared - but the actual detector values are learned.

Before running the cell, predict:

- The final loss should become very small.
- The learned kernel should become edge-like.
- The weight shape should remain `(1, 1, 1, 2)`: output channel, input channel, kernel height, kernel width.

In [ ]:
torch.manual_seed(0)

X = torch.tensor([
    [
        [1.0, 1.0, 0.0, 0.0],
        [1.0, 1.0, 0.0, 0.0],
        [1.0, 1.0, 0.0, 0.0]
    ]
]) # Shape of (1, 3, 4)

target = torch.tensor([
    [
        [0.0, 1.0, 0.0],
        [0.0, 1.0, 0.0],
        [0.0, 1.0, 0.0]
    ]
]) # Shape of (1, 3, 3)

conv = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)
optimizer = torch.optim.SGD(conv.parameters(), lr=0.3)

for step in range(120):
    pred = conv(X) # Shape of (1, 3, 3) for (channel, height, width) from channel = 1 (defined from nn.Conv2d); height = X's height of 3 - 1 + 1 = 3; widht = X' width of 4 - 1 + 1 = 3
    loss = ((pred - target) ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("final loss:", float(loss.detach()))
print("learned kernel:", conv.weight.detach().reshape(1, 2))

assert float(loss.detach()) < 1e-4
assert shape(conv.weight) == (1, 1, 1, 2) # (out_channels, in_channels, kernel_height, kernel_width)

final loss: 2.251892228244401e-09
learned kernel: tensor([[ 0.9999, -0.9999]])
torch.Size([1, 3, 3])


# 7.2.5 Break It Deliberately: Wrong Input Channel Count

`Conv2d(in_channels=1, ...)` means the layer expects each input example to have exactly one channel.

If the input tensor has three channels, the layer's kernel does not have enough channel slices to combine them.

The theory-level mistake is a mismatch between the model's expected measurement structure and the data's measurement structure.

```text
model says: each location has 1 measurement
input says: each location has 3 measurements
```

Chapter 7.4 will show how kernels handle multiple input channels correctly. Here, the goal is to make the failure recognizable.

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=2) # same as kernel_size = 2
bad_X = torch.zeros(1, 3, 4, 4) # Would match if it was (1, 1, 4, 4)

try:
    conv(bad_X)
except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0])
else:
    raise AssertionError("The channel mismatch should have failed.")

RuntimeError
Given groups=1, weight of size [1, 1, 2, 2], expected input[1, 3, 4, 4] to have 1 channels, but got 3 channels instead


# 7.2 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the chapter does not need a separate notes file.

1. Why is a kernel best understood as a local pattern detector?
> Because it slides over small regions of the input and produces a strong activation when the local values match the pattern encoded by the kernel

2. In the manual `corr2d`, what does each output value summarize?
> Each output value summarizes how well the kernel matches one local window of the input through the sum of element-wise products

3. Why does no-padding convolution shrink the spatial output?
> * The kernel cannot extend beyond the input boundaries, so it can slide over fewer positions, making the output smaller
> * Padding means adding extra pixels/values around the outside of your input before doing the convolution. It can be specified as the argument `padding` inside `nn.Conv2d`
> * Padding is important in real CNN experiments as it helps preserve the spatial dimensions and spatial information through layers

4. Why is `Conv2d` input shaped as batch, channels, height, width?
> Batch is the number of examples processed at once, channels represent different types of information (e.g. RGB has 3 channels), and height and width represent the spatial dimensions of the input

5. How did the learned-kernel cell prove that kernels are trainable parameters?
> The kernel started with random values and was updated by gradient descent until it produced an output close to the target, showing that the kernel weights are trainable parameters

6. Why did the wrong-channel input fail?
> In 7.2.5, the number of channels in the input tensor did not match the `in_channels` expected by the `Conv2d` layer